In [5]:
import torch
import torch.nn as nn

large = nn.Linear(128, 64, bias=False)
small = nn.Linear(128, 16, bias=False)

# Remove small's own weight parameter, then assign the slice as a plain tensor attribute
del small._parameters['weight']
small.weight = large.weight[:16, :]   # ✅ a view — shares storage, no copy

# Works with the standard forward() transparently:
x = torch.randn(8, 128)
y = small(x)   # calls F.linear(x, self.weight, self.bias) internally — works fine

In [3]:
y_large = large(x)
y_small = small(x)

print((y_large[:, :16] - y_small).abs().max())  # tensor(0.) — identical outputs

loss = y_large.sum() + y_small.sum()
loss.backward()

print(large.weight.grad.shape)   # [64, 128]
print(large.weight.grad[:16])    # received gradients from BOTH layers
print(large.weight.grad[16:])    # received gradients from large layer only

tensor(0., grad_fn=<MaxBackward1>)
torch.Size([64, 128])
tensor([[ 7.3875, 13.9065, -3.2458,  ..., -5.2526, 10.1608, 10.9200],
        [ 7.3875, 13.9065, -3.2458,  ..., -5.2526, 10.1608, 10.9200],
        [ 7.3875, 13.9065, -3.2458,  ..., -5.2526, 10.1608, 10.9200],
        ...,
        [ 7.3875, 13.9065, -3.2458,  ..., -5.2526, 10.1608, 10.9200],
        [ 7.3875, 13.9065, -3.2458,  ..., -5.2526, 10.1608, 10.9200],
        [ 7.3875, 13.9065, -3.2458,  ..., -5.2526, 10.1608, 10.9200]])
tensor([[ 3.6938,  6.9533, -1.6229,  ..., -2.6263,  5.0804,  5.4600],
        [ 3.6938,  6.9533, -1.6229,  ..., -2.6263,  5.0804,  5.4600],
        [ 3.6938,  6.9533, -1.6229,  ..., -2.6263,  5.0804,  5.4600],
        ...,
        [ 3.6938,  6.9533, -1.6229,  ..., -2.6263,  5.0804,  5.4600],
        [ 3.6938,  6.9533, -1.6229,  ..., -2.6263,  5.0804,  5.4600],
        [ 3.6938,  6.9533, -1.6229,  ..., -2.6263,  5.0804,  5.4600]])
